# Regresión Logística 

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve


In [13]:
# read the data

df_entrena = pd.read_csv('data/hoteles-entrena.csv')
df_prueba = pd.read_csv('data/hoteles-prueba.csv')

# print the first rows of the dataframe
print(df_entrena.head())
print(df_prueba.head())

          hotel  lead_time  stays_in_weekend_nights  stays_in_week_nights  \
0  Resort_Hotel        342                        0                     0   
1  Resort_Hotel        737                        0                     0   
2  Resort_Hotel          7                        0                     1   
3  Resort_Hotel         13                        0                     1   
4  Resort_Hotel         14                        0                     2   

   adults children meal country market_segment distribution_channel  ...  \
0       2     none   BB     PRT         Direct               Direct  ...   
1       2     none   BB     PRT         Direct               Direct  ...   
2       1     none   BB     GBR         Direct               Direct  ...   
3       1     none   BB     GBR      Corporate            Corporate  ...   
4       2     none   BB     GBR      Online_TA                TA/TO  ...   

   booking_changes  deposit_type  agent company days_in_waiting_list  \
0       

In [14]:
# Mostrar el porcentaje de valores nulos, ordenados de mayor a menor
print((df_entrena.isnull().sum() / len(df_entrena) * 100).sort_values(ascending=False))

##print(df_prueba["company"].unique())
print(df_prueba["agent"].unique())
# Eliminar las columnas con más del 50% de valores nulos (company = 92.56%, tal vez después agent= 16.31%)
df_entrena = df_entrena.drop(columns=['company'])

company                           92.565259
agent                             16.313395
country                            0.566241
hotel                              0.000000
reserved_room_type                 0.000000
total_of_special_requests          0.000000
required_car_parking_spaces        0.000000
average_daily_rate                 0.000000
customer_type                      0.000000
days_in_waiting_list               0.000000
deposit_type                       0.000000
booking_changes                    0.000000
assigned_room_type                 0.000000
previous_bookings_not_canceled     0.000000
lead_time                          0.000000
previous_cancellations             0.000000
is_repeated_guest                  0.000000
distribution_channel               0.000000
market_segment                     0.000000
meal                               0.000000
children                           0.000000
adults                             0.000000
stays_in_week_nights            

### Del EDA

In [15]:
# Convertir las fechas en el conjunto de entrenamiento y prueba
df_entrena["arrival_date"] = pd.to_datetime(df_entrena["arrival_date"])
df_prueba["arrival_date"] = pd.to_datetime(df_prueba["arrival_date"])

# Extraer el mes y el día de la semana de la columna de fecha en el conjunto de entrenamiento
df_entrena['arrival_month'] = df_entrena['arrival_date'].dt.month
df_entrena['arrival_dayofweek'] = df_entrena['arrival_date'].dt.dayofweek

# Extraer el mes y el día de la semana de la columna de fecha en el conjunto de prueba
df_prueba['arrival_month'] = df_prueba['arrival_date'].dt.month
df_prueba['arrival_dayofweek'] = df_prueba['arrival_date'].dt.dayofweek

# Función para obtener la estacionalidad según el mes
def get_season(month):
    if month in [12, 1, 2]:
        return 'Invierno'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'

# Aplicar la función de estacionalidad en ambos conjuntos
df_entrena['season'] = df_entrena['arrival_month'].apply(get_season)
df_prueba['season'] = df_prueba['arrival_month'].apply(get_season)

# Convertir la estacionalidad en variables dummy en ambos conjuntos
df_entrena = pd.get_dummies(df_entrena, columns=['season'], drop_first=True)
df_prueba = pd.get_dummies(df_prueba, columns=['season'], drop_first=True)

# Asegurarse de que el conjunto de prueba tenga las mismas columnas que el conjunto de entrenamiento
df_prueba = df_prueba.reindex(columns=df_entrena.columns, fill_value=0)

# Verificar el resultado
print(df_entrena.head())
print(df_prueba.head())



          hotel  lead_time  stays_in_weekend_nights  stays_in_week_nights  \
0  Resort_Hotel        342                        0                     0   
1  Resort_Hotel        737                        0                     0   
2  Resort_Hotel          7                        0                     1   
3  Resort_Hotel         13                        0                     1   
4  Resort_Hotel         14                        0                     2   

   adults children meal country market_segment distribution_channel  ...  \
0       2     none   BB     PRT         Direct               Direct  ...   
1       2     none   BB     PRT         Direct               Direct  ...   
2       1     none   BB     GBR         Direct               Direct  ...   
3       1     none   BB     GBR      Corporate            Corporate  ...   
4       2     none   BB     GBR      Online_TA                TA/TO  ...   

   customer_type  average_daily_rate  required_car_parking_spaces  \
0      Tran

In [16]:
# Seleccionar las columnas categóricas que queremos convertir a variables dummy
categorical_columns = ['hotel', 'meal', 'market_segment', 'distribution_channel', 
                       'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']

# Convertir las columnas categóricas en variables dummy en ambos conjuntos (entrenamiento y prueba)
df_entrena = pd.get_dummies(df_entrena, columns=categorical_columns, drop_first=True)
df_prueba = pd.get_dummies(df_prueba, columns=categorical_columns, drop_first=True)

# Asegurarse de que df_prueba tenga las mismas columnas que df_entrena
df_prueba = df_prueba.reindex(columns=df_entrena.columns, fill_value=0)

In [17]:
# Verificar que ambas tablas tengan las mismas columnas y el mismo número de columnas
print(f"Columnas en df_entrena: {df_entrena.shape[1]}")
print(f"Columnas en df_prueba: {df_prueba.shape[1]}")

# Verificar las primeras filas de df_entrena y df_prueba
print(df_entrena.head())
print(df_prueba.head())


Columnas en df_entrena: 58
Columnas en df_prueba: 58
   lead_time  stays_in_weekend_nights  stays_in_week_nights  adults children  \
0        342                        0                     0       2     none   
1        737                        0                     0       2     none   
2          7                        0                     1       1     none   
3         13                        0                     1       1     none   
4         14                        0                     2       2     none   

  country  is_repeated_guest  previous_cancellations  \
0     PRT                  0                       0   
1     PRT                  0                       0   
2     GBR                  0                       0   
3     GBR                  0                       0   
4     GBR                  0                       0   

   previous_bookings_not_canceled  booking_changes  ...  assigned_room_type_F  \
0                               0               

In [20]:
# agregar variable de tiempo total de estancia
df_entrena['total_stay'] = df_entrena['stays_in_weekend_nights'] + df_entrena['stays_in_week_nights']
df_prueba['total_stay'] = df_prueba['stays_in_weekend_nights'] + df_prueba['stays_in_week_nights']

